## Setup

In [1]:
import os
import json
import math
import random
import collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from collections import Counter, defaultdict

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# output dirs
os.makedirs('embeddings', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cpu


## Data Loading

In [2]:
# load corpus files
def load_file(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

def split_documents(text):
    # split on separator
    docs = [d.strip() for d in text.split('=' * 80) if d.strip()]
    return docs

def tokenize(text):
    # basic whitespace tokenize
    import re
    tokens = re.findall(r'[\u0600-\u06FF]+', text)
    return tokens

# load data
raw_text = load_file('raw.txt')
cleaned_text = load_file('cleaned.txt')
with open('Metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

raw_docs = split_documents(raw_text)
cleaned_docs = split_documents(cleaned_text)

print('Raw docs:', len(raw_docs))
print('Cleaned docs:', len(cleaned_docs))
print('Metadata entries:', len(metadata))

Raw docs: 300
Cleaned docs: 300
Metadata entries: 300


## Part 1 Word Embeddings

### 1.1 TF-IDF Weighting

In [3]:
# build vocabulary
VOCAB_SIZE = 10000

all_tokens = []
doc_tokens = []
for doc in cleaned_docs:
    toks = tokenize(doc)
    doc_tokens.append(toks)
    all_tokens.extend(toks)

freq = Counter(all_tokens)
vocab_words = [w for w, _ in freq.most_common(VOCAB_SIZE)]
word2idx = {w: i+1 for i, w in enumerate(vocab_words)}
word2idx['<UNK>'] = 0
idx2word = {v: k for k, v in word2idx.items()}

with open('embeddings/word2idx.json', 'w', encoding='utf-8') as f:
    json.dump(word2idx, f, ensure_ascii=False)

print('Vocab size:', len(word2idx))
print('Total tokens:', len(all_tokens))

Vocab size: 10001
Total tokens: 456492


In [4]:
# build term-document matrix
N = len(cleaned_docs)
V = len(word2idx)

# TF per document
tf_matrix = np.zeros((V, N), dtype=np.float32)
for j, toks in enumerate(doc_tokens):
    tok_count = Counter(toks)
    total = max(len(toks), 1)
    for w, c in tok_count.items():
        idx = word2idx.get(w, 0)
        tf_matrix[idx, j] = c / total

# DF per word
df = np.zeros(V, dtype=np.float32)
for j, toks in enumerate(doc_tokens):
    unique = set(word2idx.get(t, 0) for t in toks)
    for idx in unique:
        df[idx] += 1

# IDF
idf = np.log(N / (1 + df))

# TF-IDF matrix
tfidf_matrix = tf_matrix * idf[:, np.newaxis]

np.save('embeddings/tfidf_matrix.npy', tfidf_matrix)
print('TF-IDF matrix shape:', tfidf_matrix.shape)

TF-IDF matrix shape: (10001, 300)


In [5]:
# assign topic labels from metadata
TOPIC_KEYWORDS = {
    'Politics':    ['حکومت', 'وزیر', 'پارلیمان', 'انتخاب', 'سیاست', 'عمران', 'جمہوریت'],
    'Sports':      ['کرکٹ', 'کھیل', 'ٹیم', 'کھلاڑی', 'اسکور', 'میچ', 'ورلڈکپ'],
    'Economy':     ['معیشت', 'بجٹ', 'تجارت', 'بینک', 'افراط', 'روپیہ', 'ڈالر'],
    'International': ['اقوام', 'معاہدہ', 'غیر', 'یورپ', 'امریکہ', 'روس', 'یوکرین'],
    'Health':      ['صحت', 'ہسپتال', 'بیماری', 'دل', 'سیلاب', 'تعلیم', 'ویکسین']
}

doc_labels = []
for doc in cleaned_docs:
    scores = {cat: 0 for cat in TOPIC_KEYWORDS}
    toks = set(tokenize(doc))
    for cat, kws in TOPIC_KEYWORDS.items():
        scores[cat] = sum(1 for kw in kws if kw in toks)
    label = max(scores, key=scores.get)
    doc_labels.append(label)

print('Label distribution:', Counter(doc_labels))

Label distribution: Counter({'Politics': 107, 'International': 85, 'Health': 52, 'Sports': 46, 'Economy': 10})


In [6]:
# top-10 discriminative words per category
label_set = list(TOPIC_KEYWORDS.keys())
label2docs = defaultdict(list)
for j, lbl in enumerate(doc_labels):
    label2docs[lbl].append(j)

print('Top-10 words per category (TF-IDF):')
for cat in label_set:
    doc_indices = label2docs[cat]
    if not doc_indices:
        continue
    avg_tfidf = tfidf_matrix[:, doc_indices].mean(axis=1)
    top10_idx = np.argsort(avg_tfidf)[::-1][:10]
    top10_words = [idx2word.get(i, '<UNK>') for i in top10_idx]
    print(f'  {cat}: {top10_words}')

Top-10 words per category (TF-IDF):
  Politics: ['پولیس', 'اسرائیل', 'بلوچستان', 'حکومت', 'غزہ', 'شدت', 'حماس', 'سونی', 'بلوچ', 'خیبر']
  Sports: ['کرکٹ', 'میچ', 'ٹیم', 'بلے', 'بیٹ', 'غلاف', 'رنز', 'گرباز', 'چاند', 'سونیا']
  Economy: ['نوشین', 'سدرہ', 'ہیروئن', 'بیگ', 'روس', 'ارب', 'دولت', 'برکن', 'بریڈ', 'فورڈ']
  International: ['روس', 'یوکرین', 'ڈرون', 'ایران', 'اسرائیل', 'غزہ', 'روسی', 'حماس', 'ٹرمپ', 'چاند']
  Health: ['کینسر', 'بیماری', 'ڈاکٹر', 'علاج', 'دمہ', 'گلاکوما', 'دل', 'وائرس', 'اریجیت', 'پیٹرک']
